In [0]:
%pip install nltk==3.9.1 transformers torch datasets


Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
import random

class ModernChatbot:
    def __init__(self):
        self.responses = {
            "hello": ["Hi there!", "Hello!", "Greetings!"],
            "help": ["How can I assist you?", "What do you need help with?"],
            "thanks": ["You're welcome!", "Glad I could help!"],
            "bye": ["Goodbye!", "See you later!"]
        }
        
    def preprocess(self, text):
        # Convert to lowercase and tokenize
        text = text.lower()
        tokens = word_tokenize(text)
        # Remove stopwords and punctuation
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words and token.isalnum()]
        return tokens

    def generate_response(self, user_input):
        tokens = self.preprocess(user_input)
        for token in tokens:
            if token in self.responses:
                return random.choice(self.responses[token])
        return "I'm not sure how to respond to that. Could you rephrase it?"


In [0]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

class TransformerChatbot:
    def __init__(self, model_name="microsoft/DialoGPT-small"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.chat_history_ids = None
        
    def generate_response(self, user_input, max_length=1000):
        # Encode user input
        user_input_ids = self.tokenizer.encode(user_input + self.tokenizer.eos_token, 
                                             return_tensors='pt')
        
        # Generate response
        bot_output_ids = self.model.generate(
            user_input_ids,
            max_length=max_length,
            pad_token_id=self.tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=100,
            top_p=0.7,
            temperature=0.8
        )
        
        # Decode and return response
        response = self.tokenizer.decode(bot_output_ids[:, user_input_ids.shape[-1]:][0], 
                                       skip_special_tokens=True)
        return response


In [0]:
from transformers import pipeline
class AdvancedChatbot(ModernChatbot):
    def __init__(self):
        super().__init__()
        self.sentiment_analyzer = pipeline("sentiment-analysis")
        
    def analyze_sentiment(self, text):
        result = self.sentiment_analyzer(text)[0]
        return result['label'], result['score']
        
    def generate_response(self, user_input):
        # Analyze sentiment
        sentiment, confidence = self.analyze_sentiment(user_input)
        
        # Get base response
        base_response = super().generate_response(user_input)
        
        # Adjust response based on sentiment
        if sentiment == 'POSITIVE' and confidence > 0.8:
            return f"{base_response} I'm glad you're feeling positive!"
        elif sentiment == 'NEGATIVE' and confidence > 0.8:
            return f"{base_response} I notice you might be feeling down. How can I help?"
        
        return base_response


In [0]:
# Workaround for NLTK data not found error
# Monkey-patch ModernChatbot to use simple tokenization instead of NLTK
import re

original_preprocess = ModernChatbot.preprocess

def patched_preprocess(self, text):
    # Simple tokenization without NLTK
    text = text.lower()
    # Use regex to split into words
    tokens = re.findall(r'\b\w+\b', text)
    # Simple stopwords list
    stop_words = {'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", 
                  "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 
                  'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 
                  'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 
                  'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 
                  'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 
                  'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 
                  'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 
                  'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once'}
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

ModernChatbot.preprocess = patched_preprocess

def main():
    print("Choose chatbot type:")
    print("1. Rule-based")
    print("2. Transformer-based")
    choice = input("Enter choice (1/2): ")
    
    if choice == "1":
        chatbot = ModernChatbot()
    else:
        chatbot = TransformerChatbot()
    
    print("Chatbot: Hello! How can I assist you today?")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "bye":
            print("Chatbot: Goodbye!")
            break
        response = chatbot.generate_response(user_input)
        print(f"Chatbot: {response}")

if __name__ == "__main__":
    main()



Choose chatbot type:
1. Rule-based
2. Transformer-based


Enter choice (1/2):  1

Chatbot: Hello! How can I assist you today?


You:  hello there

Chatbot: Greetings!


You:  can you help me?

Chatbot: How can I assist you?


You:  What's the weather like?

Chatbot: I'm not sure how to respond to that. Could you rephrase it?


You:  bye

Chatbot: Goodbye!


The decision of whether to deploy a **Rule-Based System**, a **Transformer-Based Model**, or a **Hybrid Combination** depends on your project's balance of **predictability, context complexity, cost, and risk tolerance**.

---

### Decision Matrix: When to Use What

```
                       Is the interaction process strict, 
                     fixed, and requires 100% precision?
                                /             \
                              YES              NO
                              /                 \
                     [Rule-Based]         Does it require complex 
                                          reasoning or natural dialogue?
                                              /             \
                                            YES              NO
                                            /                 \
                                    [Transformer]        [Hybrid Architecture]
                                                       (Best for end-to-end 
                                                        enterprise systems)

```

---

### 1. When to Use Rule-Based Systems

Use a **Rule-Based Engine** when accuracy, compliance, speed, and deterministic outputs are non-negotiable.

* **Strict Regulatory & Legal Compliance:** Financial transactions, healthcare triage initial logs, tax calculations, or loan approval decision trees.
* **Exact Operational Workflows:** Multi-step form completion, appointment booking, password resets, or menu-driven interactions.
* **Zero-Tolerance for Hallucinations:** Checking order statuses, account balances, or database queries where dynamic text generation could introduce misquotes or factual errors.
* **Constrained Compute / Low Budget:** Microcontrollers, offline hardware, low-latency API triggers, or applications with zero budget for continuous GPU compute.

---

### 2. When to Use Transformer-Based Systems

Use a **Transformer / Generative Model** when flexibility, language understanding, or creative problem solving is required.

* **Open-Ended Customer Support:** Answering ambiguous, multi-part, or emotionally nuanced customer questions.
* **Unstructured Text & Knowledge Processing:** Summarizing lengthy legal contracts, analyzing feedback logs, sentiment evaluation, or translation.
* **Conversational Context Tracking:** Long-form dialogue where understanding past turns, user intent, or implied meaning is required.
* **Creative & Brainstorming Tasks:** Content writing, code drafting, ideation, and custom user coaching.

---

### 3. When to Use a Hybrid System

Use a **Hybrid Pipeline** when building production-grade enterprise products.

* **Tiered Customer Support Platforms:**
* *Rules:* Instantly match common intents, verify user authentication, and execute API actions (e.g., refund status).
* *Transformers:* Handle complex, multi-layered queries that fall outside standard FAQs.


* **Regulated Conversational AI (FinTech / Healthcare):**
* *Transformers:* Generate empathetic, natural answers based on internal documentation via Retrieval-Augmented Generation (RAG).
* *Rules:* Post-filter the model's output to strip sensitive data, verify numerical accuracy, and prevent policy violations.



---

### Summary Table

| Metric | Rule-Based | Transformer-Based | Hybrid System |
| --- | --- | --- | --- |
| **Accuracy / Determinism** | 100% Predictable | Probabilistic | Targeted 100% on logic; high flexibility elsewhere |
| **Flexibility** | Extremely Low | Extremely High | High |
| **Development Speed** | Fast initially, hard to scale | Fast with pre-trained models | Medium (requires routing architecture) |
| **Operational Cost** | Very Low | Higher (GPU/Inference costs) | Optimized (rules offload LLM calls) |
| **Ideal For** | Structured Tasks | Unstructured Understanding | Scale Enterprise Applications |

Which specific use case or industry application are you considering for implementation?